The [TensorFlow Embedding Projector](https://projector.tensorflow.org/) places
high-dimensional word vectors in a three-dimensional map where distance approximates
semantic similarity, and lets you pick a word to see its nearest neighbors. In this
assignment you build the same thing in PyTorch: you train word embeddings with
`torch.nn.Embedding`, project them to three dimensions, draw an interactive scatter, and
query the neighborhood of any token.

You will complete the parts marked with `TODO(you)`. Each raises `NotImplementedError`
until you implement it.

In [1]:
import re
from collections import Counter
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)

## Corpus and vocabulary

Word embeddings are learned from co-occurrence in text. Load a compact corpus, keep the
most frequent words as the vocabulary, and turn the text into a stream of integer ids.

In [2]:
from datasets import load_dataset

raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
text = " ".join(raw["text"]).lower()
tokens = re.findall(r"[a-z]+", text)[:300_000]
counts = Counter(tokens)

V = 8000
vocab = [w for w, _ in counts.most_common(V)]
word2idx = {w:i for i,w in enumerate(vocab)}
idx2word = {i:w for w,i in word2idx.items()}
corpus = [word2idx[w] for w in tokens if w in word2idx]

## Word2vec embeddings with softmax and cross-entropy

A word2vec model learns word embeddings by predicting context words. This is the skip-gram
architecture of word2vec: the center word predicts its context. The center word's embedding is
scored against every word in the vocabulary, a softmax turns those scores into a probability
distribution over possible context words, and the cross-entropy loss pushes up the probability
of the true context word:

$$p(o \mid c) = \frac{\exp(\mathbf{c}\cdot\mathbf{v}_o)}{\sum_{w}\exp(\mathbf{c}\cdot\mathbf{v}_w)},
\qquad L = -\log p(o \mid c).$$

The learned center embedding table is the word-vector matrix you will project.

In [3]:
class Word2Vec(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.center = nn.Embedding(vocab_size, dim)
        self.output = nn.Linear(dim, vocab_size)
        nn.init.uniform_(self.center.weight, -0.5 / dim, 0.5 / dim)

    def forward(self, center_ids):
        x = self.center(center_ids)
        return self.output(x)

In [4]:
# Build (center, context) pairs from a sliding window
window = 3
pairs = []
for i, wc in enumerate(corpus):
    for j in range(max(0, i - window), min(len(corpus), i + window + 1)):
        if j != i:
            pairs.append((wc, corpus[j]))
pairs = np.array(pairs, dtype=np.int64)

dim, B, epochs = 64, 1024, 3
model = Word2Vec(V, dim)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(epochs):
    perm = np.random.permutation(len(pairs))
    total_loss = 0.0

    for start in range(0, len(pairs), B):
        batch = pairs[perm[start:start+B]]
        center_ids = torch.tensor(batch[:,0], dtype=torch.long)
        context_ids = torch.tensor(batch[:,1], dtype=torch.long)

        opt.zero_grad()
        logits = model(center_ids)
        loss = loss_fn(logits, context_ids)
        loss.backward()
        opt.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: {total_loss:.4f}")

emb = model.center.weight.detach().cpu().numpy()

Epoch 1: 11303.1983
Epoch 2: 10860.9858
Epoch 3: 10638.0859


## Projecting the embeddings to three dimensions

The embedding matrix lives in $d=64$ dimensions. To see it, project a few thousand of the most
frequent words down to three dimensions. Principal component analysis is linear and fast; UMAP is
nonlinear and tends to separate clusters more sharply. The interactive scatter lets you rotate the
cloud and hover to read each word.

In [5]:
from sklearn.decomposition import PCA

N = 1500
plot_words = vocab[:N]
X = emb[:N]

pca3 = PCA(n_components=3).fit_transform(X)

try:
    import umap
    umap3 = umap.UMAP(n_components=3, random_state=0).fit_transform(X)
except Exception:
    umap3 = None

/home/anderson/.local/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [6]:
import plotly.graph_objects as go

def plot_embeddings(coords, words, query=None, neighbor_set=None):
    neighbor_set = neighbor_set or set()

    colors = []
    sizes = []

    for w in words:
        if w == query:
            colors.append("red")
            sizes.append(8)
        elif w in neighbor_set:
            colors.append("orange")
            sizes.append(6)
        else:
            colors.append("blue")
            sizes.append(3)

    fig = go.Figure(data=[
        go.Scatter3d(
            x=coords[:,0],
            y=coords[:,1],
            z=coords[:,2],
            mode="markers",
            text=words,
            hoverinfo="text",
            marker=dict(size=sizes, color=colors)
        )
    ])
    return fig

plot_embeddings(pca3, plot_words)

## Querying a token's neighborhood

The projector's key feature is the neighborhood query: pick a word and see its closest
words. Closeness is measured by cosine similarity in the full embedding space (not in the
3D projection). The query below returns the top-k neighbors and highlights them in the
scatter.

In [7]:
def neighbors(word, k=10):
    if word not in word2idx:
        return []

    idx = word2idx[word]

    norms = np.linalg.norm(emb, axis=1, keepdims=True)
    emb_norm = emb / np.maximum(norms, 1e-12)

    sims = emb_norm @ emb_norm[idx]
    sims[idx] = -1

    best = np.argsort(-sims)[:k]
    return [(idx2word[i], float(sims[i])) for i in best]

for w, s in neighbors("government", 10):
    print(f"{w:15s} {s:.3f}")

municipal       0.825
federal         0.823
troops          0.822
revolutionary   0.821
commonwealth    0.820
pakistani       0.813
subcontinent    0.811
courts          0.810
fledgling       0.808
invasion        0.804


In [8]:
query = "government"
nbrs = neighbors(query, 10)
neighbor_set = {w for w, _ in nbrs}

plot_embeddings(
    pca3,
    plot_words,
    query=query,
    neighbor_set=neighbor_set
)

## Exploration

Answer in the cells you add below.

1. Query several words of your choice (a few nouns, a verb, a function word). Which return clean
   semantic neighbors and which do not? Why might rare words give noisier neighbors?
2. Plot the clusters. Draw the projected embeddings (the UMAP layout separates clusters most
   clearly) and describe the groupings you see: do related words land near each other? Name a few
   clusters you can identify.

In [ ]:
words = ["house",
         "car",
         "dog",
         "play",
         "sing",
         "the",
         "and",
         "dominican"
         ]

for word in words:
    print(word)
    for w, s in neighbors(word, 10):
        print(f"{w:15s} {s:.3f}")
    print("\n")

house
coty            0.825
bradford        0.752
chancellor      0.752
judge           0.747
baptist         0.725
kit             0.719
plan            0.710
park            0.709
hall            0.708
ocosingo        0.698


car
completing      0.828
audience        0.820
page            0.819
residence       0.817
advance         0.812
submitted       0.811
bench           0.811
decision        0.807
moses           0.804
engagement      0.802


dog
cobra           0.827
odor            0.827
comb            0.825
divorced        0.816
sp              0.816
differentiation 0.812
tentatively     0.810
unity           0.809
variously       0.809
object          0.809


play
portisch        0.781
moves           0.774
beat            0.774
opening         0.772
playing         0.769
adams           0.769
personality     0.768
hartford        0.758
job             0.754
sacrifices      0.751


sing
going           0.854
get             0.850
prove           0.849
go              0.848


I queried several nouns (“house,” “car,” “dog”), verbs (“play,” “sing”), function words (“the,” “and”), and the word “dominican.” The results were mixed. The clearest semantic structure appeared for “dominican,” which strongly associated with “republic,” and for “play,” which returned related forms like “playing,” “moves,” and “beat.” I was surprised that “dominican,” despite being relatively rare, produced such a clean neighborhood; this is likely because its occurrences are dominated by the fixed phrase “Dominican Republic,” giving it a highly consistent context distribution. In general, rare words tend to produce noisier neighbors because they appear in fewer training examples, leading to fewer gradient updates and less stable estimates of their surrounding context distribution. In contrast, words like “dog” and “car” produced more diffuse and less clearly related neighbors, and “sing” mapped mostly to other common verbs such as “going,” “get,” “go,” “agree,” and “know,” suggesting weak semantic specificity. Function words like “the” and “and” showed similarly uninformative neighborhoods. Overall, the embeddings capture some meaningful semantic relationships, but many remain noisy, likely due to limited training data and training time.


In [44]:
plot_embeddings(
    umap3,
    plot_words,
)

From the UMAP visualization, related words generally appear near each other. One cluster appears to be military-related, containing words such as "navy", "fleet", "forces", "troops", "russian", "british", and "naval". Another cluster appears to be family-related, with words such as "mother", "friend", "brother", "wife", "son", "sister", and "parents" grouped together. There is also a clear chess-related cluster containing words such as "chess", "moves", "black", and "white". These clusters suggest that the Word2Vec embeddings captured meaningful semantic relationships, causing words with similar contexts to be located near one another in the projected space.